# One predictor, two missing-input rules

A fixed model makes predictions from three correlated measurements. You can make an input unavailable by
averaging over its conditional distribution given retained inputs, or by marginally replacing it.
How much does that choice change feature B's allocated improvement in squared prediction loss?

**Forecast:** conditional SAGE for B minus marginal SAGE for B, in squared-target units. The signed answer
can be any finite real number; it is not a probability or correlation and is not capped to [-1,1].
No target result or figure is prefilled. The public case was fixed before its outcome was calculated.
Build each method, pass separate practice fixtures, commit a numerical forecast and then explicitly run.

This is an exploratory analytic Gaussian toy, not independent TEST evaluation or a mastery certificate.
Exposure is `unknown`, `seen`, or `self_reported_not_seen`; a new hash never proves you are unseen.

In [ ]:
from pathlib import Path
from copy import deepcopy
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mastery_hub').is_dir())
if str(root) not in sys.path: sys.path.insert(0,str(root))
from mastery_hub.units.sage_prediction import lesson
import torch as t
from IPython.display import display, Code
t.set_num_threads(1)
CASE = deepcopy(lesson.CASE)
print(CASE)
# Existing installed environment. CPU float64, one thread; no sampling, training, download or GPU.

## Know the public inputs

Let $X=(A,B,C)$ be zero-mean Gaussian with covariance
$\Sigma=\begin{pmatrix}1&.6&-.3\\.6&1&.14\\-.3&.14&1\end{pmatrix}$.
The target is $Y=a^\top X+\epsilon$, with $a=(1,-.5,.25)$ and independent noise variance $.25$.
The fixed predictor is $f(X)=b^\top X$, with $b=(.75,.25,-.5)$. No model is fitted.
Feature names A/B/C do not imply a ranking or a sign for any contribution.

Every coalition has an integer bitmask: A is bit 0, B bit 1, C bit 2; rows 0…7 include empty and full.
For a retained set S, the restricted predictor is $c_S^\top X$. Conditional restriction uses
$c_S[S]=\Sigma_{SS}^{-1}\Sigma_{S,:}b$, zero elsewhere; marginal restriction uses $c_S[S]=b[S]$,
zero elsewhere. Empty coefficients are zero, full coefficients equal b. This covariance is positive definite,
so use `solve`, not an estimated imputer or a pseudoinverse.

The risk is $R(S)=(a-c_S)^\top\Sigma(a-c_S)+\sigma^2$ and the game is $v(S)=R(\emptyset)-R(S)$.
No sampling is needed. These are two different restriction games for the same predictor/population/loss.

## Exercise 1: restricted coefficients
Implement `restriction_coefficients(covariance, model, rule)` returning `(2**d,d)`.
For each ascending mask, select the retained indices. Conditional rule solves the covariance subproblem;
marginal rule copies the selected predictor weights. Explicitly handle empty/full. Validate shapes, finite
inputs and symmetric positive definite covariance. Never mutate inputs.
<details><summary>Help</summary>`covariance[selected][:, selected]` selects the square block;
`covariance[selected] @ model` is its right-hand side. Leave nonselected coefficients zero.
The independent practice case uses different two-dimensional inputs plus a one-dimensional check.</details>

In [ ]:
RESTRICTION_SOURCE = '''
def restriction_coefficients(covariance, model, rule):
    raise NotImplementedError("Implement this method before its practice test.")
'''
ns = lesson.method_namespace(RESTRICTION_SOURCE, complete=False)
print(lesson.practice_test(ns, 'restriction'))

Expected: `restriction: independent practice fixture passed`. No public target coefficients, risks or allocations are evaluated by this check.

## Exercise 2: risk before allocation
Implement `risk_game(coefficients, covariance, truth, noise_variance)` returning `(risks, game)`,
both shape `(2**d,)`. Compute the residual coefficient for each mask, its covariance quadratic form and
add independent noise variance. Subtract every risk from the empty risk. Do not average coefficients or games.
<details><summary>Help</summary>The quadratic contraction is over the two feature axes, preserving the
coalition axis. The practice oracle expands a separate two-dimensional quadratic by hand.</details>

In [ ]:
RISK_SOURCE = '''
def risk_game(coefficients, covariance, truth, noise_variance):
    raise NotImplementedError("Implement this method before its practice test.")
'''
ns = lesson.method_namespace(RISK_SOURCE, complete=False)
print(lesson.practice_test(ns, 'risk'))

Expected: `risk: independent practice fixture passed`. No public target coefficients, risks or allocations are evaluated by this check.

## Exercise 3: exact factorial Shapley
Implement `exact_shapley(values,n)`, returning n signed allocations from all `2**n` masks.
For i and S not containing i, weight its marginal by `factorial(k)*factorial(n-k-1)/factorial(n)`.
Keep the empty value in arbitrary practice games. The independent oracle averages all six orderings
of a different three-feature interacting game.
<details><summary>Help</summary>`mask.bit_count()` gives k. Efficiency compares the sum of allocations
with full minus empty; the formula allocates interactions without assuming the game is additive.</details>

In [ ]:
SHAPLEY_SOURCE = '''
def exact_shapley(values, n):
    raise NotImplementedError("Implement this method before its practice test.")
'''
ns = lesson.method_namespace(SHAPLEY_SOURCE, complete=False)
print(lesson.practice_test(ns, 'shapley'))

Expected: `shapley: independent practice fixture passed`. No public target coefficients, risks or allocations are evaluated by this check.

## Exercise 4: B's marginal by coalition size
Implement `marginal_by_size(values,n,player)`. Return n means, one per predecessor size k.
Average over `comb(n-1,k)` predecessor masks not containing the player. The unweighted mean of these
n strata equals that player's Shapley value; do not weight sizes by their counts again.
<details><summary>Help</summary>Insert the player with `mask | (1 << player)` and accumulate into
index `mask.bit_count()`. The practice fixture selects player 2, rather than B (index 1).</details>

In [ ]:
STRATA_SOURCE = '''
def marginal_by_size(values, n, player):
    raise NotImplementedError("Implement this method before its practice test.")
'''
ns = lesson.method_namespace(STRATA_SOURCE, complete=False)
print(lesson.practice_test(ns, 'strata'))

Expected: `strata: independent practice fixture passed`. No public target coefficients, risks or allocations are evaluated by this check.

## Exercise 5: preserve actual arrays in a JSON-native result
Implement `assemble_result(coefficients,risks,values,n,player)`. Return exactly `coefficients`, `risks`,
`game`, `shapley`, `marginals_b`. Use `.tolist()` and your own allocation/strata methods. The last field
holds the selected player's curve (B in the target, another player in practice). Do not mutate any input.
<details><summary>Help</summary>The harness keeps detached actual measurements separately, checks them
independently, and rejects an assembly that fabricates or changes coefficients, risks or game values.</details>

In [ ]:
ASSEMBLY_SOURCE = '''
def assemble_result(coefficients, risks, values, n, player):
    raise NotImplementedError("Implement this method before its practice test.")
'''
ns = lesson.method_namespace(ASSEMBLY_SOURCE + "\n\n" + SHAPLEY_SOURCE + "\n\n" + STRATA_SOURCE, complete=False)
print(lesson.practice_test(ns, 'assembly'))

Expected: `assembly: independent practice fixture passed`. No public target coefficients, risks or allocations are evaluated by this check.

## Exposed DEV ladder — can a proxy earn credit? (8–12 minutes)
Take two standard Gaussian inputs with correlation rho, truth Y=X0 and predictor f=X0.
Hand-predict B-only coefficients and B's allocation at rho=0. Then change ONLY rho to .5.
For conditional restriction, E[X0|X1]=rho X1; marginal restriction still drops X0.
Choose **A:** an unused predictor input can never earn credit; **B:** conditional access to a
correlated proxy can reduce risk. Name a B-only risk or allocation observation that rules A out.
This concerns information under restriction, not a causal intervention on the population.
All arrays below are exposed two-feature DEV, separate from the three-feature exact target.
Numeric confidence is an **unscored self-rating**, not a calibrated probability.

In [ ]:
dev_forecast_b = .5  # Deliberately rough worked forecast at rho=.5; edit before running.
dev_self_rating = .6  # unscored self-rating
dev_explanation = 'B'
dev_disconfirming_observation = 'B receives conditional credit while marginal credit stays zero.' 

In [ ]:
import matplotlib.pyplot as plt
dev_source = '\n\n'.join([RESTRICTION_SOURCE,RISK_SOURCE,SHAPLEY_SOURCE,STRATA_SOURCE,ASSEMBLY_SOURCE])
dev_ns = lesson.method_namespace(dev_source)
dev_model=t.tensor([1.,0.],dtype=t.float64);dev_truth=dev_model.clone()
def dev_proxy_game(rho, model):
    covariance=t.tensor([[1.,rho],[rho,1.]],dtype=t.float64)
    out={}
    for rule in ('conditional','marginal'):
        coeff=dev_ns['restriction_coefficients'](covariance,model,rule)
        risks,values=dev_ns['risk_game'](coeff,covariance,dev_truth,0.)
        out[rule]=(coeff,risks,dev_ns['exact_shapley'](values,2))
    return out
dev_independent=dev_proxy_game(0.,dev_model)
dev_correlated=dev_proxy_game(.5,dev_model)
dev_zero=dev_proxy_game(.5,t.zeros_like(dev_model))
fig,axes=plt.subplots(1,2,figsize=(10,3),layout='constrained')
for rule in ('conditional','marginal'):
    axes[0].plot([0,.5],[float(dev_independent[rule][2][1]),float(dev_correlated[rule][2][1])],'o-',label=rule)
    axes[1].plot(range(4),dev_correlated[rule][1].tolist(),'o-',label=rule)
axes[0].plot([0,.5],[0.,float(dev_zero['conditional'][2][1])],'--',label='Zero predictor')
axes[1].plot(range(4),dev_zero['conditional'][1].tolist(),'--',label='Zero predictor')
axes[0].set(xlabel='Correlation rho',ylabel='B SAGE (squared-target units)',title='DEV proxy allocation')
axes[1].set(xlabel='Coalition mask',ylabel='Squared-error risk',title='DEV paired restriction rules',xticks=range(4))
for ax in axes:ax.legend()
display(fig)
print('DEV B conditional/marginal credit:',float(dev_correlated['conditional'][2][1]),float(dev_correlated['marginal'][2][1]))
print('DEV B credit absolute forecast error:',abs(float(dev_correlated['conditional'][2][1])-dev_forecast_b))
assert abs(float(dev_correlated['conditional'][2][1])-.125)<1e-12
assert dev_zero['conditional'][2].count_nonzero()==0

### Worked DEV reflection / self-check
B-only conditional coefficient is .5 and risk is .75, so its empty-to-singleton gain is .25.
B adds nothing after A; averaging the two orders gives B credit .125. The rough forecast .5
has absolute error .375 squared-target units: it confused a coefficient with allocated risk reduction.
B's marginal credit stays 0; independent covariance and a zero predictor also give B zero credit.
A good self-check names the proxy route, squares rho in the risk, averages predecessor orders,
and avoids a physical-causality claim. That observed nonzero conditional credit rules A out.

### Try It Yourself — changed-condition DEV transfer
Change ONLY rho from +.5 to −.5. Predict B's coefficient AND its credit before the next cell.
Does the sign-sensitive explanation “negative correlation means negative credit” survive?
Keep truth, predictor, unit variances and restriction/loss definitions fixed.

In [ ]:
dev_transfer_forecast = {'coefficient':-.5,'credit':.125}
dev_negative=dev_proxy_game(-.5,dev_model)
dev_transfer_observed={'coefficient':float(dev_negative['conditional'][0][2,1]),
                       'credit':float(dev_negative['conditional'][2][1])}
print('DEV sign-transfer observation:',dev_transfer_observed)
print('DEV transfer absolute forecast errors:',{key:abs(dev_transfer_observed[key]-dev_transfer_forecast[key]) for key in dev_transfer_observed})
assert dev_transfer_observed == {'coefficient':-.5,'credit':.125}  # Method check, independent of forecast.
fig,ax=plt.subplots(figsize=(6,3),layout='constrained')
ax.plot([-.5,0,.5],[dev_transfer_observed['credit'],float(dev_independent['conditional'][2][1]),float(dev_correlated['conditional'][2][1])],'o-',label='Conditional B')
ax.axhline(0,color='gray',linestyle='--',label='Marginal / zero-predictor control')
ax.set(xlabel='Correlation rho',ylabel='B SAGE (squared-target units)',title='Observed DEV sign transfer');ax.legend();display(fig)
dev_revision=''  # Cite the coefficient sign change AND unchanged positive credit.

## Freeze, forecast, commit
Finish every method and its independent fixture. Freezing records exact covariance/vectors/noise, dtype,
ordered masks, method/helper/test source hashes and runtime without evaluating a target method.
Forecast the signed B contrast numerically, give confidence and a causal explanation, and name an observation
that would contradict your reasoning. Preserve prior exposure honestly. Source/runtime edits require a new
identity and commitment. Local seals are accidental-drift checks, not exam security.

In [ ]:
METHOD_SOURCE = '\n\n'.join([RESTRICTION_SOURCE, RISK_SOURCE, SHAPLEY_SOURCE, STRATA_SOURCE, ASSEMBLY_SOURCE])
identity = lesson.freeze_inputs(METHOD_SOURCE)
print('Frozen exact inputs and source; no target restriction or risk has been evaluated.')

In [ ]:
prediction = None  # Your signed conditional-minus-marginal B SAGE, squared-target units
confidence = None  # Unscored reasoning self-rating in [0,1]; not a probability
reasoning = ''  # Causal explanation and a possible disconfirming observation
prior_exposure = 'unknown'  # unknown / seen / self_reported_not_seen
commitment = lesson.commit_prediction(identity, METHOD_SOURCE, prediction, confidence, reasoning, prior_exposure)
print('Committed. No target numerical methods have run.')

## Explicit experiment, then inspect controls
Only the explicit run evaluates the public target. It repeats the separate practice fixtures and runs ten
8-coalition conditions: two restriction rules for the frozen case, identity covariance, zero predictor,
noise variance .75 and feature permutation [C,A,B]. It checks empty/full agreement, efficiency, identity-rule
equality, zero-predictor utility, the noise-risk shift with invariant credit, permutation relabeling and an
independent six-ordering allocation oracle. The covariance transfer is **not** included.

Inputs, returned arrays, assembly and runtime are checked around each student call. Failures retain actual
returned measurements and the original forecast without assigning a successful score. One cooperative
120-second CPU budget is checked between stages; a slow student body is rejected on return, not forcibly
interrupted. No model, sampling, download or GPU runs. Reference method text is optional after commitment;
adopting changed source requires freezing and committing again.

In [ ]:
# Optional implementation help after commitment; this evaluates no target:
# display(Code(lesson.reveal_methods(commitment), language='python'))

In [ ]:
try:
    receipt = lesson.run_experiment(commitment, METHOD_SOURCE)
except lesson.ExperimentFailure as failed:
    failed_record = failed.record_text
    print(lesson.replay_failure(failed_record))
    # Optional: lesson.export_record(failed_record, Path.home()/'sage-failed-attempt-1.json')
    raise  # Preserve/replay/export the failed record; no implicit retry.
observation = lesson.replay(receipt)
display(lesson.plot_receipt(receipt))
print('Observed conditional-minus-marginal B SAGE:', observation['delta_phi_b'], 'squared-target units')
print('Absolute forecast error:', observation['absolute_forecast_error'])
print('Controls:', observation['controls'])

## Explain the paired games
Read all three bars alongside B's marginal-by-size curves. Which predecessor sizes account for the contrast?
Do shared empty/full risks determine individual allocations? Which controls constrain your explanation,
and which claims do they leave open? Give a discrepancy explanation without rewriting your forecast.

The risks and allocations are exact for a fixed linear Gaussian model with known covariance. They do not
establish importance for an arbitrary nonlinear predictor, learned conditional imputer, real recipe population
or physical intervention. There is no confidence interval, independent TEST score, unseen certification or
universal feature ranking here. The original [SAGE paper](https://papers.neurips.cc/paper_files/paper/2020/hash/c7bf0b7c1a86d5eb3be2c722cf2cf746-Abstract.html)
connects this performance game to global feature attribution. After committing, compare the reference-rich
[canonical 16.4 lesson](../../../chapter16_shapley_attribution_baselines/exercises/part4_sage/16.4_SAGE_Global_Importance_exercises.ipynb),
which also separates empirical marginal replacement from conditional information.

In [ ]:
# Try It Yourself: inspect stored coalitions and another feature; no recomputation.
mask = 3  # A+B; change to any 0..7
feature = 0  # A; change to 0/1/2
for rule in ('conditional','marginal'):
    stored = lesson.replay(receipt)['results'][rule]
    print(rule, 'coefficients:', stored['coefficients'][mask], 'risk:', stored['risks'][mask],
          'game:', stored['game'][mask], 'feature SAGE:', stored['shapley'][feature])

## Forecast one changed scientific condition
Change **only** covariance to `(Sigma + I)/2`; keep truth, predictor, independent noise, loss, mask order,
allocation and numeric B contrast fixed. This changes input dependence, not a random seed. The primary run
has not evaluated this transfer. Give a signed numeric forecast, confidence and causal explanation before
creating a separate transfer commitment. A primary reflection preserves the exact primary receipt.
No second experiment is started by reflection or transfer commitment.

In [ ]:
explanation = ''  # What matched, what failed, which control mattered, and why?
transfer_prediction = None  # Numeric forecast for the covariance-only transfer
transfer_confidence = None
transfer_reasoning = ''
attempt = lesson.reflect(receipt, explanation, transfer_prediction, transfer_confidence, transfer_reasoning)
# A separate explicit commitment is required before any transfer numerical evaluation:
# transfer_identity = lesson.freeze_inputs(METHOD_SOURCE, variant='transfer')
# transfer_commitment = lesson.commit_transfer(attempt, transfer_identity, METHOD_SOURCE, prior_exposure)
# transfer_receipt = lesson.run_experiment(transfer_commitment, METHOD_SOURCE)
# display(lesson.plot_receipt(transfer_receipt))

## Keep portable records without rerunning
Exports are opt-in, create private files with mode 0600, and refuse overwrites. Replay and plots validate
stored arrays and immutable links without executing student code, solving a new target or consulting the
current runtime. Replay validation uses independent arithmetic on already stored measurements; it cannot
certify an adversarial local editor or a learner's exposure history. Failed attempts remain unscored.

In [ ]:
# Optional explicit new filenames:
# lesson.export_record(receipt, Path.home()/'sage-prediction-receipt-1.json')
# lesson.export_attempt(attempt, Path.home()/'sage-prediction-reflection-1.json')
# Later, without a new experiment:
# saved = Path.home().joinpath('sage-prediction-receipt-1.json').read_text()
# display(lesson.plot_receipt(saved))